# NFL Player Contact Detection: Blended Type Model Challenger

This notebook keeps the Notebook 5 unified model as the anchor, then blends it with the Notebook 6 type-specific models to test whether a partial specialization can improve local MCC without the private-score instability seen from full replacement.


Run modes:

- `train`: run grouped validation, tune smoothing/blend/thresholds, write reproduction artifacts, and create `submission.csv`.
- `submission`: skip validation/tuning and use `FROZEN_INFERENCE_CONFIG` to create `submission.csv` faster.


## 1. Setup and Configuration

This remains offline-safe for Kaggle. It uses the confirmed competition path, grouped validation by `game_play`, sampled negatives for training, and natural held-out play rows for threshold tuning.


In [ ]:
import gc
import json
import math
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn
from IPython.display import display
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.metrics import matthews_corrcoef, precision_recall_fscore_support
from sklearn.model_selection import GroupShuffleSplit

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 120)

NOTEBOOK_VERSION = "BLENDED_TYPE_MODELS_V2_RUN_MODE"
NOTEBOOK_NAME = "Blended Type Model Challenger"
RANDOM_STATE = 42
TARGET = "contact"
ID_COL = "contact_id"
RUN_MODE = "train"  # "train" tunes validation; "submission" uses frozen config.
RUN_VALIDATION = RUN_MODE == "train"
RUN_FINAL_SUBMISSION = True
WRITE_REPRO_ARTIFACTS = True
FROZEN_INFERENCE_CONFIG = {
    "best_window": 5,
    "best_ground_weight": 0.20,
    "best_pair_weight": 0.50,
    "best_ground_threshold": 0.59,
    "best_pair_threshold": 0.63,
}
RUN_FAST = False
FAST_SAMPLE_PLAYS = 35
NEGATIVE_TO_POSITIVE_RATIO = 6
MAX_TRAIN_ROWS = 700_000
VALID_SIZE = 0.25
THRESHOLDS = np.round(np.arange(0.05, 0.96, 0.01), 2)
SMOOTH_WINDOWS = [1, 3, 5]
BLEND_WEIGHTS = np.round(np.arange(0.0, 1.01, 0.1), 2)

REQUIRED_DATA_FILES = [
    "train_labels.csv",
    "sample_submission.csv",
    "train_player_tracking.csv",
    "test_player_tracking.csv",
]
DATA_DIR = Path("/kaggle/input/competitions/nfl-player-contact-detection")
OUTPUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")

missing_files = [
    filename for filename in REQUIRED_DATA_FILES
    if not (DATA_DIR / filename).exists()
]
if missing_files:
    raise FileNotFoundError(
        f"Missing required files in {DATA_DIR}: {missing_files}"
    )

print(f"NOTEBOOK_VERSION: {NOTEBOOK_VERSION}")
print(f"NOTEBOOK_NAME: {NOTEBOOK_NAME}")
print(f"DATA_DIR: {DATA_DIR}")
print(f"OUTPUT_DIR: {OUTPUT_DIR.resolve()}")
print(f"RUN_MODE: {RUN_MODE}")


### Version Check

When this notebook runs on Kaggle, the first code cell prints `NOTEBOOK_VERSION = BLENDED_TYPE_MODELS_V2_RUN_MODE`. If Kaggle shows an older version string, pull/sync the latest notebook before trusting the output.


## 2. Helper Functions

The main additions versus notebook 3 are nearest-player distances and probability smoothing within each contact pair sequence. Smoothing is tuned only on held-out plays.


In [ ]:
def reduce_memory_usage(df: pd.DataFrame) -> pd.DataFrame:
    """Downcast dataframe columns to reduce notebook memory usage."""
    out = df.copy()
    for col in out.columns:
        dtype = out[col].dtype
        if pd.api.types.is_integer_dtype(dtype):
            out[col] = pd.to_numeric(out[col], downcast="integer")
        elif pd.api.types.is_float_dtype(dtype):
            out[col] = pd.to_numeric(out[col], downcast="float")
        elif pd.api.types.is_object_dtype(dtype):
            nunique = out[col].nunique(dropna=False)
            if nunique / max(len(out), 1) < 0.5 and col != ID_COL:
                out[col] = out[col].astype("category")
    return out


def parse_contact_id(df: pd.DataFrame) -> pd.DataFrame:
    """Split Kaggle contact_id into play, step, player1, and player2 fields."""
    out = df.copy()
    parts = out[ID_COL].astype(str).str.split("_", expand=True)
    out["game_play"] = parts[0] + "_" + parts[1]
    out["step"] = parts[2].astype("int16")
    out["nfl_player_id_1"] = parts[3].astype(str)
    out["nfl_player_id_2"] = parts[4].astype(str)
    out["is_ground"] = out["nfl_player_id_2"].eq("G").astype("int8")
    out["contact_type"] = np.where(out["is_ground"].eq(1), "ground", "player_player")
    out["contact_pair_key"] = (
        out["game_play"].astype(str) + "_"
        + out["nfl_player_id_1"].astype(str) + "_"
        + out["nfl_player_id_2"].astype(str)
    )
    return out


def build_category_maps(*tracking_frames: pd.DataFrame) -> dict[str, dict[str, int]]:
    """Build stable integer maps for low-cardinality tracking categories."""
    maps = {}
    for col in ["position", "team"]:
        values = []
        for frame in tracking_frames:
            if col in frame.columns:
                values.extend(frame[col].astype(str).fillna("missing").unique().tolist())
        maps[col] = {value: idx for idx, value in enumerate(sorted(set(values)))}
    return maps


def add_nearest_player_features(tracking: pd.DataFrame) -> pd.DataFrame:
    """Add nearest any/team/opponent distances for each player-step."""
    required = {"game_play", "step", "nfl_player_id", "x_position", "y_position", "team"}
    if not required.issubset(tracking.columns):
        return tracking.copy()

    rows = []
    cols = ["game_play", "step", "nfl_player_id", "x_position", "y_position", "team"]
    for _, group in tracking[cols].groupby(["game_play", "step"], sort=False, observed=True):
        n = len(group)
        if n <= 1:
            part = group[["game_play", "step", "nfl_player_id"]].copy()
            part["nearest_any_distance"] = np.nan
            part["nearest_teammate_distance"] = np.nan
            part["nearest_opponent_distance"] = np.nan
            rows.append(part)
            continue

        xy = group[["x_position", "y_position"]].to_numpy(dtype="float32")
        diff = xy[:, None, :] - xy[None, :, :]
        distances = np.sqrt(np.square(diff).sum(axis=2))
        np.fill_diagonal(distances, np.inf)

        teams = group["team"].astype(str).to_numpy()
        same_team = teams[:, None] == teams[None, :]
        np.fill_diagonal(same_team, False)
        opponent = ~same_team
        np.fill_diagonal(opponent, False)

        teammate_distances = np.where(same_team, distances, np.inf)
        opponent_distances = np.where(opponent, distances, np.inf)
        part = group[["game_play", "step", "nfl_player_id"]].copy()
        part["nearest_any_distance"] = np.min(distances, axis=1)
        part["nearest_teammate_distance"] = np.min(teammate_distances, axis=1)
        part["nearest_opponent_distance"] = np.min(opponent_distances, axis=1)
        part = part.replace(np.inf, np.nan)
        rows.append(part)

    nearest = pd.concat(rows, ignore_index=True)
    out = tracking.merge(
        nearest,
        on=["game_play", "step", "nfl_player_id"],
        how="left",
    )
    return out


def prepare_tracking(tracking: pd.DataFrame, category_maps: dict[str, dict[str, int]]) -> pd.DataFrame:
    """Select, encode, and add nearest-player and prior-step dynamics."""
    keep_cols = [
        "game_play", "step", "nfl_player_id", "x_position", "y_position",
        "speed", "distance", "direction", "orientation", "acceleration", "sa",
        "position", "team", "jersey_number",
    ]
    keep_cols = [col for col in keep_cols if col in tracking.columns]
    out = tracking[keep_cols].copy()
    out["nfl_player_id"] = out["nfl_player_id"].astype(str)
    out = add_nearest_player_features(out)
    out = out.sort_values(["game_play", "nfl_player_id", "step"]).reset_index(drop=True)

    for angle_col in ["direction", "orientation"]:
        if angle_col in out.columns:
            radians = np.deg2rad(out[angle_col])
            out[f"{angle_col}_sin"] = np.sin(radians).astype("float32")
            out[f"{angle_col}_cos"] = np.cos(radians).astype("float32")

    group = out.groupby(["game_play", "nfl_player_id"], observed=True)
    for col in [
        "x_position", "y_position", "speed", "distance", "acceleration", "sa",
        "nearest_any_distance", "nearest_teammate_distance", "nearest_opponent_distance",
    ]:
        if col in out.columns:
            lag_col = f"{col}_lag1"
            out[lag_col] = group[col].shift(1)
            out[f"{col}_delta1"] = out[col] - out[lag_col]

    for col in ["direction", "orientation"]:
        if col in out.columns:
            lag = group[col].shift(1)
            diff = (out[col] - lag).abs() % 360
            out[f"{col}_delta1"] = np.minimum(diff, 360 - diff)

    for col, mapping in category_maps.items():
        if col in out.columns:
            out[f"{col}_code"] = (
                out[col].astype(str).fillna("missing").map(mapping).fillna(-1).astype("int16")
            )
        else:
            out[f"{col}_code"] = -1
    return out.drop(columns=[col for col in ["position", "team"] if col in out.columns])


def angular_difference(left: pd.Series, right: pd.Series) -> pd.Series:
    """Return smallest absolute angular difference in degrees."""
    diff = (left - right).abs() % 360
    return np.minimum(diff, 360 - diff)


def attach_tracking_features(contacts: pd.DataFrame, tracking: pd.DataFrame) -> pd.DataFrame:
    """Attach player tracking features and pairwise relative features."""
    base = contacts.copy()
    track = tracking.copy()
    player_cols = [
        col for col in track.columns
        if col not in ["game_play", "step", "nfl_player_id"]
    ]

    out = base.merge(
        track,
        left_on=["game_play", "step", "nfl_player_id_1"],
        right_on=["game_play", "step", "nfl_player_id"],
        how="left",
    ).drop(columns=["nfl_player_id"])
    out = out.rename(columns={col: f"p1_{col}" for col in player_cols})

    p2_rows = base[base["is_ground"].eq(0)].copy()
    p2 = p2_rows[[ID_COL, "game_play", "step", "nfl_player_id_2"]].merge(
        track,
        left_on=["game_play", "step", "nfl_player_id_2"],
        right_on=["game_play", "step", "nfl_player_id"],
        how="left",
    ).drop(columns=["game_play", "step", "nfl_player_id_2", "nfl_player_id"])
    p2 = p2.rename(columns={col: f"p2_{col}" for col in player_cols})
    out = out.merge(p2, on=ID_COL, how="left")

    out["dx"] = out["p1_x_position"] - out["p2_x_position"]
    out["dy"] = out["p1_y_position"] - out["p2_y_position"]
    out["pair_distance"] = np.sqrt(np.square(out["dx"]) + np.square(out["dy"]))
    out["abs_dx"] = out["dx"].abs()
    out["abs_dy"] = out["dy"].abs()

    lag_cols = [
        "p1_x_position_lag1", "p2_x_position_lag1",
        "p1_y_position_lag1", "p2_y_position_lag1",
    ]
    if all(col in out.columns for col in lag_cols):
        out["dx_lag1"] = out["p1_x_position_lag1"] - out["p2_x_position_lag1"]
        out["dy_lag1"] = out["p1_y_position_lag1"] - out["p2_y_position_lag1"]
        out["pair_distance_lag1"] = np.sqrt(
            np.square(out["dx_lag1"]) + np.square(out["dy_lag1"])
        )
        out["pair_distance_delta1"] = out["pair_distance"] - out["pair_distance_lag1"]
        out["abs_pair_distance_delta1"] = out["pair_distance_delta1"].abs()

    for col in ["speed", "distance", "acceleration", "sa"]:
        left = f"p1_{col}"
        right = f"p2_{col}"
        if left in out.columns and right in out.columns:
            out[f"{col}_diff"] = out[left] - out[right]
            out[f"abs_{col}_diff"] = out[f"{col}_diff"].abs()

    for col in ["nearest_any_distance", "nearest_teammate_distance", "nearest_opponent_distance"]:
        left = f"p1_{col}"
        right = f"p2_{col}"
        if left in out.columns and right in out.columns:
            out[f"{col}_min_pair"] = np.minimum(out[left], out[right])

    for col in ["direction", "orientation"]:
        left = f"p1_{col}"
        right = f"p2_{col}"
        if left in out.columns and right in out.columns:
            out[f"{col}_angle_diff"] = angular_difference(out[left], out[right])

    if "p1_team_code" in out.columns and "p2_team_code" in out.columns:
        out["same_team"] = (out["p1_team_code"] == out["p2_team_code"]).astype("float32")

    out["step_float"] = out["step"].astype("float32")
    return out


def sample_training_rows(labels: pd.DataFrame) -> pd.DataFrame:
    """Keep all positives and a controlled number of negatives for training."""
    if RUN_FAST:
        labels = labels[labels["game_play"].isin(
            pd.Series(labels["game_play"].unique()).sample(
                min(FAST_SAMPLE_PLAYS, labels["game_play"].nunique()),
                random_state=RANDOM_STATE,
            )
        )].copy()

    positives = labels[labels[TARGET].eq(1)]
    negatives = labels[labels[TARGET].eq(0)]
    max_negatives = min(
        len(negatives),
        max(len(positives) * NEGATIVE_TO_POSITIVE_RATIO, MAX_TRAIN_ROWS - len(positives)),
    )
    negatives = negatives.sample(max_negatives, random_state=RANDOM_STATE)
    sampled = pd.concat([positives, negatives], ignore_index=True)
    if len(sampled) > MAX_TRAIN_ROWS:
        sampled = sampled.sample(MAX_TRAIN_ROWS, random_state=RANDOM_STATE)
    return sampled.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)


def get_feature_columns(df: pd.DataFrame) -> list[str]:
    """Return numeric model features, excluding IDs and target columns."""
    excluded = {
        ID_COL, TARGET, "game_play", "nfl_player_id_1", "nfl_player_id_2",
        "contact_type", "contact_pair_key", "datetime",
    }
    return [
        col for col in df.columns
        if col not in excluded and pd.api.types.is_numeric_dtype(df[col])
    ]


def score_thresholds(y_true: np.ndarray, probabilities: np.ndarray) -> pd.DataFrame:
    """Score probability thresholds with MCC and positive rate."""
    rows = []
    for threshold in THRESHOLDS:
        pred = (probabilities >= threshold).astype("int8")
        precision, recall, f1, _ = precision_recall_fscore_support(
            y_true, pred, average="binary", zero_division=0
        )
        rows.append({
            "threshold": threshold,
            "mcc": matthews_corrcoef(y_true, pred),
            "positive_rate": pred.mean(),
            "precision": precision,
            "recall": recall,
            "f1": f1,
        })
    return pd.DataFrame(rows).sort_values("mcc", ascending=False)


def smooth_probabilities(scored: pd.DataFrame, window: int) -> np.ndarray:
    """Smooth probabilities within each play/pair sequence."""
    if window <= 1:
        return scored["probability"].to_numpy()
    ordered = scored.sort_values(["contact_pair_key", "step"]).copy()
    ordered["smoothed"] = (
        ordered.groupby("contact_pair_key", observed=True)["probability"]
        .transform(lambda s: s.rolling(window, center=True, min_periods=1).mean())
    )
    return ordered.sort_index()["smoothed"].to_numpy()


def make_model():
    """Create the strongest offline-safe sklearn model available."""
    try:
        return HistGradientBoostingClassifier(
            learning_rate=0.055,
            max_iter=280,
            max_leaf_nodes=39,
            l2_regularization=0.04,
            early_stopping=True,
            validation_fraction=0.12,
            random_state=RANDOM_STATE,
        )
    except TypeError:
        return RandomForestClassifier(
            n_estimators=280,
            min_samples_leaf=8,
            class_weight="balanced_subsample",
            n_jobs=-1,
            random_state=RANDOM_STATE,
        )


def write_json_artifact(filename: str, payload: dict) -> None:
    """Write a compact JSON artifact to OUTPUT_DIR."""
    if not WRITE_REPRO_ARTIFACTS:
        return
    path = OUTPUT_DIR / filename
    with open(path, "w") as fp:
        json.dump(payload, fp, indent=2, default=str)
    print(f"wrote: {path}")


def frozen_config_value(key: str):
    """Read an inference parameter from the frozen submission config."""
    return FROZEN_INFERENCE_CONFIG[key]


## 3. Load Data and Prepare Tracking


In [ ]:
labels = reduce_memory_usage(pd.read_csv(
    DATA_DIR / "train_labels.csv",
    parse_dates=["datetime"],
))
sample_submission = pd.read_csv(DATA_DIR / "sample_submission.csv")
train_tracking = reduce_memory_usage(pd.read_csv(
    DATA_DIR / "train_player_tracking.csv",
    parse_dates=["datetime"],
))
test_tracking = reduce_memory_usage(pd.read_csv(
    DATA_DIR / "test_player_tracking.csv",
    parse_dates=["datetime"],
))

labels = parse_contact_id(labels)
sample_submission = parse_contact_id(sample_submission)
category_maps = build_category_maps(train_tracking, test_tracking)
train_tracking_prepared = prepare_tracking(train_tracking, category_maps)
test_tracking_prepared = prepare_tracking(test_tracking, category_maps)

print(f"labels: {labels.shape}")
print(f"sample_submission: {sample_submission.shape}")
print(f"train_tracking_prepared: {train_tracking_prepared.shape}")
print(f"test_tracking_prepared: {test_tracking_prepared.shape}")
print(f"positive rate: {labels[TARGET].mean():.5f}")
labels.head()


## 4. Grouped Split and Feature Construction

Training uses all positives plus sampled negatives from training plays. Validation uses natural held-out play rows so MCC threshold tuning sees the real class balance.


In [ ]:
if RUN_VALIDATION:
    model_source = labels.copy()
    if RUN_FAST:
        sampled_plays = pd.Series(model_source["game_play"].unique()).sample(
            min(FAST_SAMPLE_PLAYS, model_source["game_play"].nunique()),
            random_state=RANDOM_STATE,
        )
        model_source = model_source[model_source["game_play"].isin(sampled_plays)].copy()

    splitter = GroupShuffleSplit(
        n_splits=1,
        test_size=VALID_SIZE,
        random_state=RANDOM_STATE,
    )
    train_source_idx, valid_source_idx = next(
        splitter.split(model_source, model_source[TARGET], groups=model_source["game_play"])
    )
    train_source = model_source.iloc[train_source_idx].copy()
    valid_rows = model_source.iloc[valid_source_idx].copy()
    train_rows = sample_training_rows(train_source)

    print(f"sampled training rows: {train_rows.shape}")
    print(f"natural validation rows: {valid_rows.shape}")
    print(f"sampled training positive rate: {train_rows[TARGET].mean():.5f}")
    print(f"natural validation positive rate: {valid_rows[TARGET].mean():.5f}")
    display(train_rows.groupby("contact_type", observed=True)[TARGET].agg(["count", "mean", "sum"]))
    display(valid_rows.groupby("contact_type", observed=True)[TARGET].agg(["count", "mean", "sum"]))

    train_features = attach_tracking_features(train_rows, train_tracking_prepared)
    valid_features = attach_tracking_features(valid_rows, train_tracking_prepared)
    feature_cols = get_feature_columns(train_features)

    print(f"train feature frame: {train_features.shape}")
    print(f"valid feature frame: {valid_features.shape}")
    print(f"feature count: {len(feature_cols)}")
    display(train_features[feature_cols].head())

    del model_source, train_source, train_rows, valid_rows
    gc.collect()

else:
    print("RUN_MODE=submission: skipping validation split and local tuning.")
    train_features = None
    valid_features = None
    feature_cols = None


## 5. Train Unified and Type-Specific Models


In [ ]:
if RUN_VALIDATION:
    X_train = train_features[feature_cols].replace([np.inf, -np.inf], np.nan)
    y_train = train_features[TARGET].astype("int8")
    X_valid = valid_features[feature_cols].replace([np.inf, -np.inf], np.nan)
    y_valid = valid_features[TARGET].astype("int8")
    valid_meta = valid_features[[
        ID_COL, "game_play", "step", "contact_type", "contact_pair_key",
    ]].copy()

    unified_model = make_model()
    unified_model.fit(X_train, y_train)
    unified_valid_prob = unified_model.predict_proba(X_valid)[:, 1]
    print(f"unified model: {type(unified_model).__name__}")

    models = {}
    type_valid_prob = np.zeros(len(X_valid), dtype="float32")
    for contact_type in ["ground", "player_player"]:
        train_mask = train_features["contact_type"].eq(contact_type).to_numpy()
        valid_mask = valid_features["contact_type"].eq(contact_type).to_numpy()
        model = make_model()
        model.fit(X_train.loc[train_mask], y_train.loc[train_mask])
        type_valid_prob[valid_mask] = model.predict_proba(X_valid.loc[valid_mask])[:, 1]
        models[contact_type] = model
        print(
            f"{contact_type} model: {type(model).__name__}; "
            f"train rows: {train_mask.sum():,}; "
            f"valid rows: {valid_mask.sum():,}; "
            f"train positive rate: {y_train.loc[train_mask].mean():.5f}; "
            f"valid positive rate: {y_valid.loc[valid_mask].mean():.5f}"
        )

    print(f"combined train rows: {len(X_train):,}")
    print(f"combined valid rows: {len(X_valid):,}")
    print(f"combined train positive rate: {y_train.mean():.5f}")
    print(f"combined valid positive rate: {y_valid.mean():.5f}")

else:
    print("RUN_MODE=submission: skipping validation model training.")
    unified_valid_prob = None
    type_valid_prob = None
    valid_meta = None
    y_valid = None


## 6. Tune Blend, Smoothing, and Type-Specific Thresholds

Search blend weights separately for ground and player-player rows. A weight of `0.0` means the Notebook 5 unified model; a weight of `1.0` means the Notebook 6 type-specific model. The challenger should only be trusted if it beats Notebook 5 locally and improves or preserves the ground slice.


In [ ]:
def mcc_from_counts(tp: int, tn: int, fp: int, fn: int) -> float:
    """Compute MCC from confusion counts without row-level predictions."""
    denominator = math.sqrt(
        float(tp + fp)
        * float(tp + fn)
        * float(tn + fp)
        * float(tn + fn)
    )
    if denominator == 0.0:
        return 0.0
    return float((tp * tn) - (fp * fn)) / denominator


def threshold_confusion_counts(y_true: np.ndarray, probabilities: np.ndarray) -> pd.DataFrame:
    """Return confusion counts for every configured threshold."""
    rows = []
    y_true = y_true.astype("int8")
    for threshold in THRESHOLDS:
        pred = (probabilities >= threshold).astype("int8")
        tp = int(((pred == 1) & (y_true == 1)).sum())
        fp = int(((pred == 1) & (y_true == 0)).sum())
        tn = int(((pred == 0) & (y_true == 0)).sum())
        fn = int(((pred == 0) & (y_true == 1)).sum())
        rows.append({
            "threshold": threshold,
            "tp": tp,
            "fp": fp,
            "tn": tn,
            "fn": fn,
        })
    return pd.DataFrame(rows)


def score_type_thresholds(scored: pd.DataFrame, probabilities: np.ndarray) -> pd.DataFrame:
    """Score separate ground and player-player threshold pairs efficiently."""
    work = scored[["y_true", "contact_type"]].copy()
    work["probability"] = probabilities
    ground = work[work["contact_type"].eq("ground")]
    pair = work[work["contact_type"].eq("player_player")]
    ground_counts = threshold_confusion_counts(
        ground["y_true"].to_numpy(),
        ground["probability"].to_numpy(),
    )
    pair_counts = threshold_confusion_counts(
        pair["y_true"].to_numpy(),
        pair["probability"].to_numpy(),
    )

    rows = []
    for ground_row in ground_counts.itertuples(index=False):
        for pair_row in pair_counts.itertuples(index=False):
            tp = ground_row.tp + pair_row.tp
            fp = ground_row.fp + pair_row.fp
            tn = ground_row.tn + pair_row.tn
            fn = ground_row.fn + pair_row.fn
            positive_rate = (tp + fp) / max(tp + fp + tn + fn, 1)
            rows.append({
                "ground_threshold": ground_row.threshold,
                "pair_threshold": pair_row.threshold,
                "mcc": mcc_from_counts(tp, tn, fp, fn),
                "positive_rate": positive_rate,
            })
    return pd.DataFrame(rows).sort_values("mcc", ascending=False)


def apply_type_thresholds(
    scored: pd.DataFrame,
    probabilities: np.ndarray,
    ground_threshold: float,
    pair_threshold: float,
) -> np.ndarray:
    """Apply contact-type-specific thresholds to probabilities."""
    thresholds = np.where(
        scored["contact_type"].eq("ground"),
        ground_threshold,
        pair_threshold,
    )
    return (probabilities >= thresholds).astype("int8")



def blend_probabilities(
    scored: pd.DataFrame,
    unified_probabilities: np.ndarray,
    type_probabilities: np.ndarray,
    ground_weight: float,
    pair_weight: float,
) -> np.ndarray:
    """Blend unified and type-specific probabilities by contact type."""
    weights = np.where(
        scored["contact_type"].eq("ground"),
        ground_weight,
        pair_weight,
    )
    return (
        (1.0 - weights) * unified_probabilities
        + weights * type_probabilities
    ).astype("float32")


if RUN_VALIDATION:
    valid_scored = valid_meta.assign(
        y_true=y_valid.to_numpy(),
    )
    all_scores = []
    top_scores = []
    for ground_weight in BLEND_WEIGHTS:
        for pair_weight in BLEND_WEIGHTS:
            blended_prob = blend_probabilities(
                valid_scored,
                unified_valid_prob,
                type_valid_prob,
                ground_weight,
                pair_weight,
            )
            blend_scored = valid_scored.assign(probability=blended_prob)
            for window in SMOOTH_WINDOWS:
                prob = smooth_probabilities(blend_scored, window)
                global_scores = score_thresholds(valid_scored["y_true"].to_numpy(), prob)
                type_scores = score_type_thresholds(valid_scored, prob)
                best_type = type_scores.iloc[0].to_dict()
                best_global = global_scores.iloc[0].to_dict()
                best_type["smooth_window"] = window
                best_type["ground_blend_weight"] = ground_weight
                best_type["pair_blend_weight"] = pair_weight
                best_type["global_threshold"] = best_global["threshold"]
                best_type["global_mcc"] = best_global["mcc"]
                best_type["global_positive_rate"] = best_global["positive_rate"]
                all_scores.append(best_type)
                top_scores.append(type_scores.head(3).assign(
                    smooth_window=window,
                    ground_blend_weight=ground_weight,
                    pair_blend_weight=pair_weight,
                ))

    blend_scores = pd.DataFrame(all_scores).sort_values("mcc", ascending=False)
    best_window = int(blend_scores.iloc[0]["smooth_window"])
    best_ground_weight = float(blend_scores.iloc[0]["ground_blend_weight"])
    best_pair_weight = float(blend_scores.iloc[0]["pair_blend_weight"])
    best_ground_threshold = float(blend_scores.iloc[0]["ground_threshold"])
    best_pair_threshold = float(blend_scores.iloc[0]["pair_threshold"])
    valid_blended_prob = blend_probabilities(
        valid_scored,
        unified_valid_prob,
        type_valid_prob,
        best_ground_weight,
        best_pair_weight,
    )
    valid_blend_scored = valid_scored.assign(probability=valid_blended_prob)
    valid_best_prob = smooth_probabilities(valid_blend_scored, best_window)
    valid_pred = apply_type_thresholds(
        valid_scored,
        valid_best_prob,
        best_ground_threshold,
        best_pair_threshold,
    )

    print(f"best smoothing window: {best_window}")
    print(f"best ground blend weight: {best_ground_weight:.2f}")
    print(f"best player-player blend weight: {best_pair_weight:.2f}")
    print(f"best ground threshold: {best_ground_threshold:.2f}")
    print(f"best player-player threshold: {best_pair_threshold:.2f}")
    print(f"best local MCC: {matthews_corrcoef(valid_scored['y_true'], valid_pred):.6f}")
    display(blend_scores.head(20))
    display(pd.concat(top_scores, ignore_index=True).sort_values("mcc", ascending=False).head(20))

    valid_scored = valid_scored.assign(
        smoothed_probability=valid_best_prob,
        prediction=valid_pred,
    )
    slice_scores = []
    for contact_type, part in valid_scored.groupby("contact_type", observed=True):
        slice_scores.append({
            "contact_type": contact_type,
            "rows": len(part),
            "actual_rate": part["y_true"].mean(),
            "predicted_rate": part["prediction"].mean(),
            "mcc": matthews_corrcoef(part["y_true"], part["prediction"]),
        })
    slice_scores_df = pd.DataFrame(slice_scores)
    display(slice_scores_df)


    if WRITE_REPRO_ARTIFACTS:
        blend_scores.to_csv(OUTPUT_DIR / "threshold_search.csv", index=False)
        slice_scores_df.to_csv(OUTPUT_DIR / "validation_summary.csv", index=False)
        write_json_artifact("model_config.json", {
            "notebook_version": NOTEBOOK_VERSION,
            "run_mode": RUN_MODE,
            "random_state": RANDOM_STATE,
            "max_train_rows": MAX_TRAIN_ROWS,
            "negative_to_positive_ratio": NEGATIVE_TO_POSITIVE_RATIO,
            "best_window": best_window,
            "best_ground_weight": best_ground_weight,
            "best_pair_weight": best_pair_weight,
            "best_ground_threshold": best_ground_threshold,
            "best_pair_threshold": best_pair_threshold,
            "local_mcc": float(matthews_corrcoef(valid_scored["y_true"], valid_pred)),
        })
        print(f"wrote: {OUTPUT_DIR / 'threshold_search.csv'}")
        print(f"wrote: {OUTPUT_DIR / 'validation_summary.csv'}")

else:
    print("RUN_MODE=submission: using frozen inference config.")
    best_window = int(frozen_config_value("best_window"))
    best_ground_weight = float(frozen_config_value("best_ground_weight"))
    best_pair_weight = float(frozen_config_value("best_pair_weight"))
    best_ground_threshold = float(frozen_config_value("best_ground_threshold"))
    best_pair_threshold = float(frozen_config_value("best_pair_threshold"))
    print(f"best smoothing window: {best_window}")
    print(f"best ground blend weight: {best_ground_weight:.2f}")
    print(f"best player-player blend weight: {best_pair_weight:.2f}")
    print(f"best ground threshold: {best_ground_threshold:.2f}")
    print(f"best player-player threshold: {best_pair_threshold:.2f}")


## 7. Train Final Blended Models and Create Submission

The final submission trains both the unified model and type-specific models, blends their probabilities using validation-tuned weights, smooths by contact-pair sequence, and applies type-specific thresholds.


In [ ]:
final_train_rows = sample_training_rows(labels)
final_features = attach_tracking_features(final_train_rows, train_tracking_prepared)
if feature_cols is None:
    feature_cols = get_feature_columns(final_features)
    print(f"feature count: {len(feature_cols)}")

X_full = final_features[feature_cols].replace([np.inf, -np.inf], np.nan)
y_full = final_features[TARGET].astype("int8")

final_unified_model = make_model()
final_unified_model.fit(X_full, y_full)
print(f"final unified model: {type(final_unified_model).__name__}; rows: {len(X_full):,}")

final_type_models = {}
for contact_type in ["ground", "player_player"]:
    train_mask = final_features["contact_type"].eq(contact_type).to_numpy()
    model = make_model()
    model.fit(X_full.loc[train_mask], y_full.loc[train_mask])
    final_type_models[contact_type] = model
    print(
        f"final {contact_type} model: {type(model).__name__}; "
        f"rows: {train_mask.sum():,}; "
        f"positive rate: {y_full.loc[train_mask].mean():.5f}"
    )

submission_rows = sample_submission[[
    ID_COL, "game_play", "step", "nfl_player_id_1", "nfl_player_id_2",
    "is_ground", "contact_type", "contact_pair_key",
]].copy()
test_features = attach_tracking_features(submission_rows, test_tracking_prepared)
X_test = test_features[feature_cols].replace([np.inf, -np.inf], np.nan)
unified_test_prob = final_unified_model.predict_proba(X_test)[:, 1]
type_test_prob = np.zeros(len(X_test), dtype="float32")
for contact_type, model in final_type_models.items():
    test_mask = test_features["contact_type"].eq(contact_type).to_numpy()
    if test_mask.sum() == 0:
        continue
    type_test_prob[test_mask] = model.predict_proba(X_test.loc[test_mask])[:, 1]

test_scored = submission_rows[[ID_COL, "contact_pair_key", "contact_type", "step"]].copy()
test_scored["probability"] = blend_probabilities(
    test_scored,
    unified_test_prob,
    type_test_prob,
    best_ground_weight,
    best_pair_weight,
)
test_scored["smoothed_probability"] = smooth_probabilities(test_scored, best_window)
submission_pred = apply_type_thresholds(
    test_scored,
    test_scored["smoothed_probability"].to_numpy(),
    best_ground_threshold,
    best_pair_threshold,
)

submission = sample_submission[[ID_COL]].copy()
submission = submission.merge(
    test_scored[[ID_COL, "smoothed_probability"]],
    on=ID_COL,
    how="left",
)
submission[TARGET] = submission_pred
submission = submission[[ID_COL, TARGET]]
output_path = OUTPUT_DIR / "submission.csv"
submission.to_csv(output_path, index=False)

print(f"wrote: {output_path}")
print(f"submission shape: {submission.shape}")
print(f"predicted positive rate: {submission[TARGET].mean():.5f}")
display(submission.head())


if WRITE_REPRO_ARTIFACTS:
    write_json_artifact("feature_columns.json", {"feature_columns": feature_cols})
    write_json_artifact("run_metadata.json", {
        "notebook_version": NOTEBOOK_VERSION,
        "notebook_name": NOTEBOOK_NAME,
        "run_mode": RUN_MODE,
        "random_state": RANDOM_STATE,
        "sklearn_version": sklearn.__version__,
        "numpy_version": np.__version__,
        "pandas_version": pd.__version__,
        "train_rows": int(len(X_full)),
        "feature_count": int(len(feature_cols)),
        "best_window": int(best_window),
        "best_ground_weight": float(best_ground_weight),
        "best_pair_weight": float(best_pair_weight),
        "best_ground_threshold": float(best_ground_threshold),
        "best_pair_threshold": float(best_pair_threshold),
        "submission_rows": int(len(submission)),
        "submission_positive_rate": float(submission[TARGET].mean()),
    })

del final_train_rows, final_features, X_full, X_test, test_features
gc.collect()


## 8. Decision Rule

Submit this challenger only if blended modeling improves grouped validation over Notebook 5's `0.67650` local MCC and does not reduce the ground-contact MCC below `0.50623`. If it does not clear both gates, keep Notebook 5 as the selected champion.
